In [1]:
import numpy as np
import scanpy as sc
import spapros as sp

import anndata as ad

adata_full = ad.read_h5ad("/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/abc_atlas.h5ad", backed='r')

# Number of cells you want
N = 300_000

# Total number of cells in the dataset
total_cells = adata_full.n_obs
print("Total cells =", total_cells)

# Randomly sample 400k unique indices
np.random.seed(42)  # reproducibility
idx = np.random.choice(total_cells, size=N, replace=False)

# IMPORTANT: sort indices for efficient slicing in backed mode
idx_sorted = np.sort(idx)

# Load only those cells into memory
adata = adata_full[idx_sorted, :].to_memory()

print(adata)

Total cells = 2349544
AnnData object with n_obs × n_vars = 300000 × 32285
    obs: 'abc_sample_id', 'anatomical_division_label', 'barcoded_cell_sample_label', 'brain_section_label', 'cell_barcode', 'class', 'class_color', 'cluster', 'cluster_alias', 'cluster_color', 'dataset_label', 'donor_genotype', 'donor_label', 'donor_sex', 'entity', 'feature_matrix_label', 'library_label', 'library_method', 'neurotransmitter', 'neurotransmitter_color', 'region_of_interest_acronym', 'region_of_interest_color', 'region_of_interest_order', 'subclass', 'subclass_color', 'supertype', 'supertype_color', 'x', 'y'


In [2]:
import pandas as pd

# Count cells per class
counts = adata.obs['class'].value_counts(dropna=False)

# Convert to DataFrame
df = counts.reset_index()
df.columns = ["class", "count"]

# Extract numeric prefix (if exists), otherwise set to very large number to push NaN to the end
df["class_num"] = (
    df["class"]
    .astype(str)
    .str.extract(r"^(\d+)", expand=False)
    .fillna("999")        # ensures NaN or missing numbers come LAST
    .astype(int)
)

# Sort by numeric prefix
df_sorted = df.sort_values("class_num")

df_sorted_clean = df_sorted.reset_index(drop=True)
print(df_sorted_clean[["class", "count"]])


# Number of unique classes
print("\nNumber of classes:", df["class"].nunique())


                class  count
0       01 IT-ET Glut  34436
1   02 NP-CT-L6b Glut   9886
2       03 OB-CR Glut    326
3      04 DG-IMN Glut   2013
4      05 OB-IMN GABA   6790
5     06 CTX-CGE GABA   4777
6     07 CTX-MGE GABA   3995
7     08 CNU-MGE GABA   1998
8     09 CNU-LGE GABA  13809
9         10 LSX GABA   3874
10    11 CNU-HYa GABA   8683
11         12 HY GABA   5582
12    13 CNU-HYa Glut   4131
13         14 HY Glut   4398
14   15 HY Gnrh1 Glut     25
15      16 HY MM Glut   1076
16      17 MH-LH Glut    831
17         18 TH Glut   5872
18         19 MB Glut  12756
19         20 MB GABA   8161
20         21 MB Dopa    557
21      22 MB-HB Sero    341
22          23 P Glut   3242
23         24 MY Glut   3329
24     25 Pineal Glut     18
25          26 P GABA   2498
26         27 MY GABA   4300
27         28 CB GABA   6467
28         29 CB Glut  17930
29      30 Astro-Epen  37927
30       31 OPC-Oligo  60972
31             32 OEC     87
32        33 Vascular  16550
33          34

In [4]:
# ----------------------------
# 1. Normalize and log1p
# ----------------------------
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

# ----------------------------
# 2. HVG selection
# ----------------------------
sc.pp.highly_variable_genes(adata, n_top_genes=32000)

# ------------------
# Save HVG panel
# ------------------
# Seleziona solo i geni HVG
hvg_df = adata.var[adata.var["highly_variable"]].copy()

# Se non esiste 'highly_variable_rank', calcoliamo il rank dallo score
if "highly_variable_rank" in hvg_df.columns:
    hvg_df["HVG_rank"] = hvg_df["highly_variable_rank"]
else:
    # Rank: gene con HVG_score più alto = rank 1
    hvg_df["HVG_rank"] = hvg_df["dispersions_norm"].rank(
        method="dense",
        ascending=False
    )

# HVG_score = dispersions_norm
hvg_df["HVG_score"] = hvg_df["dispersions_norm"]

# Crea la tabella finale
panel = hvg_df[["HVG_rank", "HVG_score"]].copy()
panel.insert(0, "Ensembl_ID", hvg_df.index)

# Ordina per rank
panel = panel.sort_values("HVG_rank")

# Salva il CSV
panel.to_csv("outputs/score_panels/HVG_panel.csv", index=False)

panel.head()

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: `n_top_genes` > number of normalized dispersions, returning all genes with normalized dispersions.
  return fn(*args_all, **kw)


,Ensembl_ID,HVG_rank,HVG_score
gene_identifier,,,
ENSMUSG00000007097,ENSMUSG00000007097,1.0,6.043343
ENSMUSG00000026473,ENSMUSG00000026473,2.0,6.004214
ENSMUSG00000028565,ENSMUSG00000028565,3.0,5.750812
ENSMUSG00000064373,ENSMUSG00000064373,4.0,5.658700
ENSMUSG00000055643,ENSMUSG00000055643,5.0,5.491703


In [6]:
# ----------------------------
# 3. PCA
# ----------------------------
sc.tl.pca(adata)

# Loadings: genes × PCs
loadings = adata.varm["PCs"]       # shape (n_genes, n_pcs)
genes = adata.var_names            # Ensembl IDs

# Varianza spiegata da ogni PC
var_ratio = adata.uns["pca"]["variance_ratio"]   # length n_pcs

# Assicuriamoci che le dimensioni combacino
n_pcs = loadings.shape[1]
var_ratio = var_ratio[:n_pcs]

# Calcola gli score pesati di ogni PC per ogni gene
# score_gene_pc = loading_gene_pc * var_ratio_pc
weighted_scores = loadings * var_ratio

# Score combinato per gene = somma pesata su tutte le PC
combined_score = np.sum(np.abs(weighted_scores), axis=1)

# Crea DataFrame
df = pd.DataFrame({"Ensembl_ID": genes})

# Aggiungi le colonne PC1_loading, PC2_loading, ...
for i in range(n_pcs):
    df[f"PC{i+1}_loading"] = loadings[:, i]

# Aggiungi colonne PC1_score, PC2_score, ...
for i in range(n_pcs):
    df[f"PC{i+1}_score"] = weighted_scores[:, i]

# Aggiungi combined score
df["PCA_combined_score"] = combined_score

# Ranking dei geni
df["PCA_combined_rank"] = df["PCA_combined_score"].rank(
    method="dense", ascending=False
)

# Ordina per importanza
df = df.sort_values("PCA_combined_rank")

# Salva CSV
df.to_csv("outputs/score_panels/PCA_genes_full_panel.csv", index=False)

df.head()




/tmp/ipykernel_1126232/482042434.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"PC{i+1}_score"] = weighted_scores[:, i]
/tmp/ipykernel_1126232/482042434.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["PCA_combined_score"] = combined_score
/tmp/ipykernel_1126232/482042434.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragme

,Ensembl_ID,PC1_loading,PC2_loading,PC3_loading,PC4_loading,PC5_loading,PC6_loading,PC7_loading,PC8_loading,PC9_loading,...,PC43_score,PC44_score,PC45_score,PC46_score,PC47_score,PC48_score,PC49_score,PC50_score,PCA_combined_score,PCA_combined_rank
1513,ENSMUSG00000007097,0.026947,-0.024154,0.028266,-0.012243,-0.010564,0.016968,0.004609,-0.022377,-0.021870,...,-6.114467e-06,-1.603679e-05,5.553826e-06,5.027389e-06,-6.592474e-06,-8.222123e-06,-2.078617e-06,3.282867e-06,0.004176,1.0
15676,ENSMUSG00000046240,0.035181,0.005628,0.048350,0.008323,0.007928,0.009965,0.010084,-0.004494,-0.011483,...,-2.626919e-06,4.092735e-07,-5.518538e-07,5.563645e-06,-4.579184e-06,-5.340440e-06,-5.608402e-06,6.827220e-07,0.004076,2.0
27035,ENSMUSG00000046160,0.026947,0.025289,0.027779,0.010418,0.012183,0.031661,-0.000862,-0.022960,-0.021847,...,-2.245643e-06,6.922571e-07,-6.453493e-07,4.240179e-06,-2.099494e-06,8.549711e-07,9.421803e-08,2.260945e-06,0.004054,3.0
28205,ENSMUSG00000040260,0.032065,0.008120,0.032606,0.013645,-0.008296,-0.033883,0.004700,0.010896,0.006864,...,-5.464194e-06,-1.099109e-07,1.433610e-06,3.554394e-06,9.137203e-08,-1.083350e-06,-9.081779e-07,4.027322e-06,0.004041,4.0
4387,ENSMUSG00000105265,0.022280,0.016195,0.036131,0.013266,0.015179,0.041802,0.004881,-0.000959,0.029542,...,6.807592e-07,-1.412118e-06,-1.082727e-05,-7.910763e-07,5.784104e-06,-1.606621e-06,2.439478e-06,5.476936e-06,0.004009,5.0


In [7]:
# ============================================================
# CONFIGURATION FLAGS (CHOOSE A, B, or C)
# ============================================================

# Choose the grouping mode:
# A = Leiden
# B = Louvain
# C = Existing annotation in adata.obs
mode = "A"  

# Parameters for mode A
leiden_resolution = 0.6
leiden_key = f"leiden_{leiden_resolution}"

# Parameters for mode B
louvain_key = "louvain"

# Parameters for mode C
existing_annotation_key = "cell_type"   # <-- MUST exist in adata.obs


# ============================================================
# STEP 1 — SELECT GROUPING METHOD
# ============================================================

if mode == "A":
    print(f"Using **Leiden** clustering (resolution = {leiden_resolution})")
    sc.pp.neighbors(adata, n_pcs=50)
    sc.tl.leiden(adata, resolution=leiden_resolution, key_added=leiden_key)
    group_key = leiden_key

elif mode == "B":
    print("Using **Louvain** clustering")
    sc.pp.neighbors(adata, n_pcs=50)
    sc.tl.louvain(adata, key_added=louvain_key)
    group_key = louvain_key

elif mode == "C":
    print(f"Using existing annotation: {existing_annotation_key}")
    if existing_annotation_key not in adata.obs.columns:
        raise ValueError(f"Column '{existing_annotation_key}' not found in adata.obs")
    group_key = existing_annotation_key

else:
    raise ValueError("Mode must be 'A', 'B', or 'C'.")


print(f"\n➡️  Grouping key selected: {group_key}")
print(f"Groups found: {adata.obs[group_key].unique()}\n")


# ============================================================
# STEP 2 — DIFFERENTIAL EXPRESSION
# ============================================================

sc.tl.rank_genes_groups(
    adata,
    groupby=group_key,
    method="wilcoxon",
    n_genes=adata.n_vars
)

de = adata.uns["rank_genes_groups"]
clusters = de["names"].dtype.names


# ============================================================
# STEP 3 — BUILD GENE × CLUSTER DE SCORE TABLE
# ============================================================

DE_scores = pd.DataFrame(index=adata.var_names)

for cl in clusters:
    DE_scores[cl] = de["scores"][cl]


# ============================================================
# STEP 4 — GLOBAL DE SCORE (max across clusters)
# ============================================================

DE_global_score = DE_scores.max(axis=1)
DE_global_rank = DE_global_score.rank(method="dense", ascending=False)


# ============================================================
# STEP 5 — FINAL PANEL + CSV EXPORT
# ============================================================

panel = pd.DataFrame({
    "Ensembl_ID": adata.var_names,
    "DE_global_score": DE_global_score,
    "DE_global_rank": DE_global_rank
})

panel = panel.sort_values("outputs/score_panels/DE_global_rank")

csv_name = f"DE_panel_{group_key}.csv"
panel.to_csv(csv_name, index=False)

print(f"✅ Saved DE panel to: {csv_name}")
print(panel.head())


Using **Leiden** clustering (resolution = 0.6)


/tmp/ipykernel_1126232/4023387245.py:29: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=leiden_resolution, key_added=leiden_key)



➡️  Grouping key selected: leiden_0.6
Groups found: ['0', '32', '38', '14', '22', ..., '33', '15', '30', '39', '36']
Length: 40
Categories (40, object): ['0', '1', '2', '3', ..., '36', '37', '38', '39']



/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:458: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:460: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/scanpy/tools/_ran

✅ Saved DE panel to: DE_panel_leiden_0.6.csv
                            Ensembl_ID  DE_global_score  DE_global_rank
gene_identifier                                                        
ENSMUSG00000051951  ENSMUSG00000051951       240.864273             1.0
ENSMUSG00000089699  ENSMUSG00000089699       240.803177             2.0
ENSMUSG00000102331  ENSMUSG00000102331       239.657928             3.0
ENSMUSG00000102343  ENSMUSG00000102343       239.509598             4.0
ENSMUSG00000025900  ENSMUSG00000025900       239.429962             5.0


In [16]:

DE_res.keys(), PCA_res.keys()


(Index(['selection', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10',
        '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22',
        '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34',
        '35', '36', '37', '38', '39', '40', '41', '42'],
       dtype='object'),
 Index(['selection', 'selection_score', 'selection_ranking'], dtype='object'))

In [20]:
# HVG gene list
# Filtra solo gli HVG e ordina per dispersione normalizzata
hvg_sorted = (
    adata.var.loc[adata.var["highly_variable"]]
    .sort_values("dispersions_norm", ascending=False)
)

for N in [1000, 2000, 4000,8000, 16000]:
    topN_hvg = hvg_sorted.index[:N].tolist()
    save_panel_csv(topN_hvg, f"outputs/HVG_top{N}.csv")

    # Top-N DE genes
    top_de_genes = DE_rank.index[:N].tolist()
    save_panel_csv(
        top_de_genes,
        f"outputs/DE_panel_top{N}.csv"
    )

    # Top-N PCA genes
    top_pca_genes = PCA_rank.index[:N].tolist()
    save_panel_csv(
        top_pca_genes,
        f"outputs/PCA_panel_top{N}.csv"
    )


In [3]:
def build_standard_panels(adata, celltype_key="celltype"):
    """
    Standard Spapros panels: PCA + DE, different n_pca_genes and panel sizes.
    Returns a dict: {panel_name: gene_list}
    """
    panels = {}

    # We choose a small grid for n_pca_genes
    PCA_GENE_COUNTS = [300]  # low, medium, high emphasis on PCA genes

    for n_genes in PANEL_SIZES:  # 500, 750, 1000
        for n_pca_genes in PCA_GENE_COUNTS:
            name = f"standard_n{n_genes}_pca{n_pca_genes}"
            selector = sp.se.ProbesetSelector(
                adata,
                n=n_genes,
                n_pca_genes=n_pca_genes,
                celltype_key=celltype_key,
                verbosity=1,
                save_dir=None,
            )
            selector.select_probeset()
            panel_genes = selector.probeset[selector.probeset["selection"]].index.tolist()
            panels[name] = panel_genes

            filename = f"outputs/standard_n{n_genes}_pca{n_pca_genes}.csv"
            save_panel_csv(panel_genes, filename)

    return panels

standard_panels = build_standard_panels(adata, celltype_key)


/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:1482: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = a.obs.groupby(ct_key)["test_set"].value_counts()


Note: The following celltypes' test set sizes for forest training are below min_test_n (=20):
	 03 OB-CR Glut   : 5
	 06 CTX-CGE GABA : 3
	 07 CTX-MGE GABA : 3
	 08 CNU-MGE GABA : 13
	 12 HY GABA      : 2
	 14 HY Glut      : 1
	 20 MB GABA      : 17
	 23 P Glut       : 6
	 26 P GABA       : 4
The genes selected for those cell types potentially don't generalize well. Find the genes for each of those cell types in self.genes_of_primary_trees after running self.select_probeset().


Output()

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:458: 
PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 
times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a 
de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:460: 
PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 
times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a 
de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]

...

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 06 CTX-CGE GABA in train or test set. Celltype 06 CTX-CGE GABA is not included 
as reference celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 12 HY GABA in train or test set. Celltype 12 HY GABA is not included as 
reference celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 14 HY Glut in train or test set. Celltype 14 HY Glut is not included as 
reference celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [14]:
def build_DE_heavy_panels(DE_rank):
    """
    DE-heavy panels: take top DE genes directly.
    """
    panels = {}

    # DE_rank is a sorted Series (ascending rank => more important)
    all_genes_sorted_by_DE = DE_rank.index.tolist()

    for n_genes in PANEL_SIZES:
        name = f"DE_heavy_n{n_genes}"
        panel_genes = all_genes_sorted_by_DE[:n_genes]
        panels[name] = panel_genes

        panel_genes = all_genes_sorted_by_DE[:n_genes]
        filename = f"outputs/DE_heavy_n{n_genes}.csv"
        save_panel_csv(panel_genes, filename)


    return panels

DE_heavy_panels = build_DE_heavy_panels(DE_rank)


In [ ]:
def build_dual_run_panels(adata, celltype_key="celltype"):
    """
    Dual-run SpaPRos panels, modified to perform 4 rounds of probe selection:
    - Round 1: DE-only (no PCA)
    - Rounds 2–4: PCA+DE on remaining genes
    Each round selects n_genes/4 genes, giving total n_genes.
    """

    panels = {}

    for n_genes in PANEL_SIZES:
        name = f"dual_run_n{n_genes}"

        # number of genes per round
        per_round = n_genes // 4

        # -------------------------
        # ROUND 1 — DE only
        # -------------------------
        selector1 = sp.se.ProbesetSelector(
            adata,
            n=per_round,
            n_pca_genes=0,
            celltype_key=celltype_key,
            verbosity=1,
            save_dir=None,
        )
        selector1.select_probeset()
        sel1 = selector1.probeset[selector1.probeset["selection"]].index.tolist()

        selected_genes = sel1.copy()

        # Prepare remaining genes
        remaining = [g for g in adata.var_names if g not in selected_genes]
        adata_rem = adata[:, remaining]

        # -------------------------
        # ROUNDS 2–4 — PCA+DE
        # -------------------------
        for round_idx in range(2, 5):
            selector = sp.se.ProbesetSelector(
                adata_rem,
                n=per_round,
                n_pca_genes=150,
                celltype_key=celltype_key,
                verbosity=1,
                save_dir=None,
            )
            selector.select_probeset()
            sel = selector.probeset[selector.probeset["selection"]].index.tolist()

            selected_genes.extend(sel)

            # Update remaining set
            remaining = [g for g in adata_rem.var_names if g not in sel]
            adata_rem = adata_rem[:, remaining]

        # -------------------------
        # Finalize panel
        # -------------------------
        panels[name] = selected_genes

        filename = f"outputs/{name}.csv"
        save_panel_csv(selected_genes, filename)

    return panels


# run it
dual_run_panels = build_dual_run_panels(adata, celltype_key)



Note: No PCA selection will be performed since n_pca_genes = 0. The selected genes will only be based on the DE forests. In that case it can happen that fewer than n = 250 genes are selected. To get n = 250 exclude the selected genes of the first run from adata, rerun the method, and combine the results of the two runs.



/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:1482: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = a.obs.groupby(ct_key)["test_set"].value_counts()


Note: The following celltypes' test set sizes for forest training are below min_test_n (=20):
	 51 : 5
The genes selected for those cell types potentially don't generalize well. Find the genes for each of those cell types in self.genes_of_primary_trees after running self.select_probeset().


Output()

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:458: 
PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 
times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a 
de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]

...

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:463: 
PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 
times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a 
de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "pvals"] = pvals[global_indices]

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:473: 
PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 
times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a 
de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "pvals_adj"] = pvals_adj[global_indices]

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:484: 
PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 
times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a 
de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "logfoldchanges"] = np.log2(

/p/software/default/stages/2025/software/Python-bundle-PyPI/2024.06-GCCcore-13.3.0/lib/python3.12/site-packages/job
lib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the 
executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(

/p/software/default/stages/2025/software/Python-bundle-PyPI/2024.06-GCCcore-13.3.0/lib/python3.12/site-packages/job
lib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the 
executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(

In [ ]:
def build_gene_removed_panels_generic(adata, celltype_key="celltype", PCA_rank=None):
    """
    Panels where we remove some genes (e.g. top PCA) before running standard Spapros.
    """
    panels = {}

    if PCA_rank is None:
        raise ValueError("PCA_rank must be provided as a sorted Series.")

    # Take, for example, the 500 strongest PCA genes and remove them
    genes_to_remove = PCA_rank.index[:500].tolist()

    remaining_genes = [g for g in adata.var_names if g not in genes_to_remove]
    adata2 = adata[:, remaining_genes]

    for n_genes in PANEL_SIZES:
        name = f"removed_topPCA_n{n_genes}"
        selector = sp.se.ProbesetSelector(
            adata2,
            n=n_genes,
            n_pca_genes=150,
            celltype_key=celltype_key,
            verbosity=1,
            save_dir=None,
        )
        selector.select_probeset()
        panel_genes = selector.probeset[selector.probeset["selection"]].index.tolist()
        panels[name] = panel_genes
        filename = f"outputs/removed_run_n{n_genes}.csv"
        save_panel_csv(panel_genes, filename)

    return panels

generic_removed_panels = build_gene_removed_panels_generic(
    adata,
    celltype_key,
    PCA_rank=PCA_rank,
)


In [ ]:
def build_gene_removed_panels_important(adata, celltype_key="celltype",
                                        DE_rank=None, PCA_rank=None,
                                        top_k=300):
    """
    Remove 'very important HVGs' (top_k intersect of DE & PCA) and then run Spapros.
    """
    panels = {}

    if DE_rank is None or PCA_rank is None:
        raise ValueError("DE_rank and PCA_rank must be provided as sorted Series.")

    # Take top_k DE and top_k PCA
    top_DE_genes = set(DE_rank.index[:top_k].tolist())
    top_PCA_genes = set(PCA_rank.index[:top_k].tolist())

    # 'Very important HVGs' = intersection
    important_genes = top_DE_genes.intersection(top_PCA_genes)

    # Remove them from the gene universe
    remaining_genes = [g for g in adata.var_names if g not in important_genes]
    adata2 = adata[:, remaining_genes]

    for n_genes in PANEL_SIZES:
        name = f"removed_importantHVGs_n{n_genes}_top{top_k}"
        selector = sp.se.ProbesetSelector(
            adata2,
            n=n_genes,
            n_pca_genes=150,
            celltype_key=celltype_key,
            verbosity=1,
            save_dir=None,
        )
        selector.select_probeset()
        panel_genes = selector.probeset[selector.probeset["selection"]].index.tolist()
        panels[name] = panel_genes

    return panels

important_removed_panels = build_gene_removed_panels_important(
    adata,
    celltype_key,
    DE_rank=DE_rank,
    PCA_rank=PCA_rank,
    top_k=300,
)


In [ ]:
#Removing specifically for some cell type the best gene

In [ ]:
all_panels = {}
all_panels.update(standard_panels)
all_panels.update(marker_heavy_panels)
all_panels.update(DE_heavy_panels)
all_panels.update(constraint_panels)
all_panels.update(dual_run_panels)
all_panels.update(generic_removed_panels)
all_panels.update(important_removed_panels)

len(all_panels)  # number of distinct panels you generated
